In [1]:
import pandas as pd
from pathlib import Path

In [6]:
demand_df = pd.read_parquet("../data/processed/hourly_demand.parquet")

demand_df.head()

,PULocationID,pickup_hour_timestamp,demand,date,hour,day_of_week,month,is_weekend
0,1,2026-01-01 05:00:00,1,2026-01-01,5,3,1,0
1,1,2026-01-01 11:00:00,1,2026-01-01,11,3,1,0
2,1,2026-01-01 12:00:00,1,2026-01-01,12,3,1,0
3,1,2026-01-01 15:00:00,2,2026-01-01,15,3,1,0
4,1,2026-01-01 17:00:00,2,2026-01-01,17,3,1,0


In [7]:
# Sort data by pickup location and timestamp
# This is necessary before creating lag and rolling features
demand_df = demand_df.sort_values(
    by=["PULocationID", "pickup_hour_timestamp"]
)

# Reset index after sorting
demand_df = demand_df.reset_index(drop=True)

# Preview the sorted data
demand_df.head()

,PULocationID,pickup_hour_timestamp,demand,date,hour,day_of_week,month,is_weekend
0,1,2026-01-01 05:00:00,1,2026-01-01,5,3,1,0
1,1,2026-01-01 11:00:00,1,2026-01-01,11,3,1,0
2,1,2026-01-01 12:00:00,1,2026-01-01,12,3,1,0
3,1,2026-01-01 15:00:00,2,2026-01-01,15,3,1,0
4,1,2026-01-01 17:00:00,2,2026-01-01,17,3,1,0


In [ ]:
# Create 1-hour lag demand feature
# Previous hour's demand for the same pickup location


demand_df["lag_1"] = (demand_df.groupby("PULocationID")["demand"].shift(1))

In [ ]:
# Create 24-hour lag demand feature
# Demand at the same hour on the previous day


demand_df["lag_24"] = (demand_df.groupby("PULocationID")["demand".shift(24))

In [ ]:
# Preview lag features


demand_df[
    [
        "PULocationID",
        "pickup_hour_timestamp",
        "demand",
        "lag_1",
        "lag_24",
    ]
].head(30)

,PULocationID,pickup_hour_timestamp,demand,lag_1,lag_24
0,1,2026-01-01 05:00:00,1,NaN,NaN
1,1,2026-01-01 11:00:00,1,1.0,NaN
2,1,2026-01-01 12:00:00,1,1.0,NaN
3,1,2026-01-01 15:00:00,2,1.0,NaN
4,1,2026-01-01 17:00:00,2,2.0,NaN
5,1,2026-01-01 21:00:00,1,2.0,NaN
6,1,2026-01-01 22:00:00,1,1.0,NaN
7,1,2026-01-01 23:00:00,1,1.0,NaN
8,1,2026-01-02 04:00:00,2,1.0,NaN
9,1,2026-01-02 05:00:00,1,2.0,NaN


In [ ]:
# Create 3-hour rolling average demand
# Average demand over the previous 3 hours for each pickup location


demand_df["rolling_mean_3"] = (demand_df.groupby("PULocationID")["demand"].transform(lambda x: x.rolling(window=3).mean()))

In [ ]:
# Create 24-hour rolling average demand
# Average demand over the previous 24 hours


demand_df["rolling_mean_24"] = (demand_df.groupby("PULocationID")["demand"].transform(lambda x: x.rolling(window=24).mean()))

In [ ]:
# Create 24-hour rolling standard deviation
# Measures demand variability over the previous 24 hours


demand_df["rolling_std_24"] = (demand_df.groupby("PULocationID")["demand"].transform(lambda x: x.rolling(window=24).std()))

In [15]:
# Preview newly created features
demand_df[
    [
        "PULocationID",
        "pickup_hour_timestamp",
        "demand",
        "lag_1",
        "lag_24",
        "rolling_mean_3",
        "rolling_mean_24",
        "rolling_std_24"
    ]
].head(30)

,PULocationID,pickup_hour_timestamp,demand,lag_1,lag_24,rolling_mean_3,rolling_mean_24,rolling_std_24
0,1,2026-01-01 05:00:00,1,NaN,NaN,NaN,NaN,NaN
1,1,2026-01-01 11:00:00,1,1.0,NaN,NaN,NaN,NaN
2,1,2026-01-01 12:00:00,1,1.0,NaN,1.000000,NaN,NaN
3,1,2026-01-01 15:00:00,2,1.0,NaN,1.333333,NaN,NaN
4,1,2026-01-01 17:00:00,2,2.0,NaN,1.666667,NaN,NaN
5,1,2026-01-01 21:00:00,1,2.0,NaN,1.666667,NaN,NaN
6,1,2026-01-01 22:00:00,1,1.0,NaN,1.333333,NaN,NaN
7,1,2026-01-01 23:00:00,1,1.0,NaN,1.000000,NaN,NaN
8,1,2026-01-02 04:00:00,2,1.0,NaN,1.333333,NaN,NaN
9,1,2026-01-02 05:00:00,1,2.0,NaN,1.333333,NaN,NaN


In [ ]:
# Create Peak Hour feature
# Peak hours: 7-10 AM and 4-8 PM

demand_df["is_peak_hour"] = demand_df["hour"].isin([7, 8, 9, 10, 16, 17, 18, 19, 20]).astype(int)

In [17]:
# Check missing values created due to lag and rolling operations

demand_df.isnull().sum()

PULocationID                0
pickup_hour_timestamp       0
demand                      0
date                        0
hour                        0
day_of_week                 0
month                       0
is_weekend                  0
lag_1                     262
lag_24                   6052
rolling_mean_3            524
rolling_mean_24          5809
rolling_std_24           5809
is_peak_hour                0
dtype: int64

In [18]:
# Remove rows containing missing values
# These occur because lag and rolling features cannot be computed
# for the first few observations of each pickup location

demand_df = demand_df.dropna()

# Reset index
demand_df = demand_df.reset_index(drop=True)

In [19]:
# Verify the cleaned dataset

demand_df.info()

demand_df.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 456253 entries, 0 to 456252
Data columns (total 14 columns):
 #   Column                 Non-Null Count   Dtype         
---  ------                 --------------   -----         
 0   PULocationID           456253 non-null  int32         
 1   pickup_hour_timestamp  456253 non-null  datetime64[us]
 2   demand                 456253 non-null  int64         
 3   date                   456253 non-null  object        
 4   hour                   456253 non-null  int32         
 5   day_of_week            456253 non-null  int32         
 6   month                  456253 non-null  int32         
 7   is_weekend             456253 non-null  int32         
 8   lag_1                  456253 non-null  float64       
 9   lag_24                 456253 non-null  float64       
 10  rolling_mean_3         456253 non-null  float64       
 11  rolling_mean_24        456253 non-null  float64       
 12  rolling_std_24         456253 non-null  floa

,PULocationID,pickup_hour_timestamp,demand,date,hour,day_of_week,month,is_weekend,lag_1,lag_24,rolling_mean_3,rolling_mean_24,rolling_std_24,is_peak_hour
0,1,2026-01-04 18:00:00,1,2026-01-04,18,6,1,1,2.0,1.0,2.666667,1.416667,0.880547,1
1,1,2026-01-04 19:00:00,1,2026-01-04,19,6,1,1,1.0,1.0,1.333333,1.416667,0.880547,1
2,1,2026-01-04 23:00:00,1,2026-01-04,23,6,1,1,1.0,1.0,1.000000,1.416667,0.880547,0
3,1,2026-01-05 08:00:00,1,2026-01-05,8,0,1,0,1.0,2.0,1.000000,1.375000,0.875388,1
4,1,2026-01-05 12:00:00,1,2026-01-05,12,0,1,0,1.0,2.0,1.000000,1.333333,0.868115,0


In [20]:
# Save the feature-engineered dataset

demand_df.to_parquet("../data/processed/feature_engineered_data.parquet",index=False)

# 04. Feature Engineering

## Objectives
- Create meaningful features from the hourly demand dataset.
- Capture temporal demand patterns.
- Improve the predictive power of machine learning models.
- Prepare the final dataset for model training.

---

## Steps Performed

### 1. Loaded Hourly Demand Dataset
- Imported the aggregated hourly demand dataset.
- Verified dataset dimensions.

---

### 2. Sorted the Dataset
Sorted by:

- Pickup Location ID
- Pickup Hour Timestamp

This ensures lag and rolling features are computed in chronological order.

---

### 3. Created Lag Features

Generated historical demand features:

- **Lag 1**
  - Demand during the previous hour.

- **Lag 24**
  - Demand during the same hour on the previous day.

These help the model learn short-term and daily demand patterns.

---

### 4. Created Rolling Features

Generated moving statistical features for each pickup location:

- Rolling Mean (3 Hours)
- Rolling Mean (24 Hours)
- Rolling Standard Deviation (24 Hours)

These capture local demand trends and variability.

---

### 5. Created Peak Hour Feature

Created a binary feature:

- `is_peak_hour`

Peak Hours:

- 7 AM – 10 AM
- 4 PM – 8 PM

This helps the model recognize rush-hour traffic demand.

---

### 6. Handled Missing Values

Lag and rolling operations create missing values for the first few observations of each pickup location.

Performed:

- Removed rows containing missing values.
- Reset the DataFrame index.

---

### 7. Verified the Dataset

Checked:

- Data types
- Dataset information
- Sample records

Ensured all engineered features were successfully created.

---

### 8. Saved Final Dataset

Exported the feature-engineered dataset as:

```
data/processed/feature_engineered_data.parquet
```

---

## Features Available for Model Training

### Target Variable
- Demand

### Location Feature
- Pickup Location ID

### Temporal Features
- Hour
- Day of Week
- Month
- Weekend Indicator
- Peak Hour Indicator

### Historical Features
- Lag 1
- Lag 24

### Rolling Statistics
- Rolling Mean (3 Hours)
- Rolling Mean (24 Hours)
- Rolling Standard Deviation (24 Hours)

---

## Output

A fully feature-engineered dataset ready for machine learning model training and evaluation.

# Understanding Lag Features and Rolling Features

## The Problem

Suppose we want to answer the following question:

> **"How many taxis will be needed in Zone 50 at 8 PM today?"**

Can the machine learning model use the demand at **8 PM**?

**No.**

That is the value we are trying to predict.

Instead, the model can only use information that is already available before 8 PM, such as:

- Demand at 7 PM
- Demand at 6 PM
- Demand at 8 PM yesterday
- Average demand over the last few hours

These historical values become our features.

---

# Example Dataset

| Time | Demand |
|------|--------|
| 5 PM | 18 |
| 6 PM | 20 |
| 7 PM | 25 |
| 8 PM | ? |

Our goal is to predict the demand at **8 PM**.

---

# Lag Features

A **lag feature** simply means looking back at previous observations.

Instead of giving the model future information, we provide historical information.

---

## Lag 1

**Lag 1** means the demand from **one hour earlier**.

| Time | Demand | Lag 1 |
|------|--------|--------|
| 5 PM | 18 | NaN |
| 6 PM | 20 | 18 |
| 7 PM | 25 | 20 |
| 8 PM | ? | 25 |

When predicting the demand at **8 PM**, the model receives:

> Previous hour demand = **25**

This becomes one of the input features.

---

## Lag 24

Since our data is hourly, **Lag 24** means the demand **24 hours earlier**, i.e., the same hour on the previous day.

| Time | Demand | Lag 24 |
|------|--------|---------|
| Yesterday 8 PM | 31 | - |
| Today 8 PM | ? | 31 |

When predicting today's 8 PM demand, the model receives:

> Demand yesterday at 8 PM = **31**

Taxi demand often follows daily patterns, so this feature is highly informative.

---

# Why Do Lag Features Work?

Taxi demand is not random.

For example:

### Monday

| Time | Demand |
|------|--------|
| 6 PM | 90 |
| 7 PM | 110 |
| 8 PM | 120 |

### Tuesday

| Time | Demand |
|------|--------|
| 6 PM | 95 |
| 7 PM | 115 |
| 8 PM | ? |

If demand has been increasing recently, there is a good chance it will remain high.

The model learns these repeating patterns from the lag features.

---

# Rolling Mean

Instead of looking at only one previous hour, we can summarize several previous hours.

Example:

| Hour | Demand |
|------|--------|
| 5 PM | 15 |
| 6 PM | 20 |
| 7 PM | 25 |

Rolling Mean (3 Hours):

```
(15 + 20 + 25) / 3 = 20
```

Instead of giving the model only one previous value, we provide:

> Average demand over the previous 3 hours = **20**

This smooths sudden spikes and gives the model a better understanding of recent trends.

---

# Rolling Standard Deviation

Rolling Standard Deviation measures how much demand changes over time.

### Stable Demand

```
20
21
20
19
20
```

Variation is very small.

Rolling Standard Deviation is **low**.

---

### Unstable Demand

```
5
40
10
35
8
```

Variation is much larger.

Rolling Standard Deviation is **high**.

This tells the model whether demand has been stable or highly volatile.

---

# Why Group by Pickup Location?

Consider two pickup zones.

### Zone A

| Hour | Demand |
|------|--------|
| 5 PM | 12 |
| 6 PM | 15 |
| 7 PM | 18 |

### Zone B

| Hour | Demand |
|------|--------|
| 5 PM | 80 |
| 6 PM | 95 |
| 7 PM | 100 |

If we did **not** group by `PULocationID`, the previous demand for Zone B could incorrectly come from Zone A.

That would produce meaningless features.

Therefore, we calculate lag features independently for every pickup location.

```python
demand_df.groupby("PULocationID")["demand"].shift(1)
```

This ensures that each pickup location maintains its own historical demand sequence.

---

# Example Feature Vector

Suppose we want to predict demand for **Zone 120 at 3 PM**.

The model receives the following features:

| Feature | Value |
|----------|------:|
| Hour | 15 |
| Day of Week | Tuesday |
| Weekend | 0 |
| Lag 1 | 82 |
| Lag 24 | 77 |
| Rolling Mean (3) | 80 |
| Rolling Mean (24) | 74 |
| Rolling Standard Deviation (24) | 6.2 |
| Peak Hour | 1 |

Using all these features together, the model predicts:

> **Expected Demand = 85 taxi trips**

---

# Key Takeaway

Feature engineering transforms raw taxi trip records into meaningful historical information.

Instead of asking the model to predict demand with only the current date and time, we provide historical demand patterns such as:

- Previous hour demand (Lag 1)
- Previous day demand (Lag 24)
- Short-term demand trends (Rolling Mean)
- Demand variability (Rolling Standard Deviation)

These engineered features allow the machine learning model to learn temporal patterns and make accurate demand forecasts.